# 02. Layer 1 và Layer 2

Embed BGE-M3 (nên bật GPU), tính s và p, grid search λ×τ trên dev theo từng tenant, báo cáo trên test.

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", "https://github.com/vinhlee9x/conflict-detection-obsolescence-forecasting.git", "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not os.path.exists("src") and os.getcwd() != "/":
        os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Thư mục làm việc:", os.getcwd())

In [ ]:
# Tuỳ chọn: giữ data/ và artifacts/ trên Google Drive giữa các session Colab
USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    target = "/content/drive/MyDrive/conflict-detection"
    for folder in ("data/processed", "artifacts"):
        os.makedirs(f"{target}/{folder}", exist_ok=True)
        subprocess.run(f"cp -rn {folder}/. {target}/{folder}/ && rm -rf {folder} && ln -s {target}/{folder} {folder}", shell=True, check=True)

In [ ]:
from src import pipeline
import pandas as pd
summary = pipeline.embed("kb", backend="bge-m3")
summary

## ViNLI: cosine có tách được mâu thuẫn không?

In [ ]:
pipeline.vinli(backend="bge-m3", per_label=1000)

## Calibration và ablation

In [ ]:
results = pipeline.layer2("kb")
pd.DataFrame(results["test"])[["scope", "method_label", "lambda", "tau", "precision", "recall", "f1", "f1_ci95", "pr_auc"]]

In [ ]:
pd.DataFrame(results["test_by_variant"])

In [ ]:
{tenant: {k: cfg[k] for k in ("FUSION_LAMBDA", "CONFLICT_THRESHOLD_TAU", "dev_f1")} for tenant, cfg in results["config"]["tenants"].items()}

## Luồng end-to-end: tài liệu mới → candidate → xác nhận

Lấy 3 tài liệu ban hành gần nhất làm ví dụ; người dùng xác nhận candidate đầu tiên.

In [ ]:
from src.conflict import confirm_conflicts, detect_candidates
from src.graph import KnowledgeGraph
from src.io_utils import read_parquet
from src.paths import DATA_PROCESSED
docs = read_parquet(DATA_PROCESSED / "kb_docs.parquet")
edges = read_parquet(DATA_PROCESSED / "kb_edges.parquet")
new_ids = docs.sort_values("issued_at").doc_id.tail(3).tolist()
graph = KnowledgeGraph(docs.doc_id.tolist(), edges)
candidates = detect_candidates(new_ids, docs, pipeline.load_vectors("kb"), graph, results["config"], top_k=15)
candidates

In [ ]:
confirm_conflicts(graph, candidates.head(1))
graph.edges[graph.edges.type == "CONFLICTS_WITH"]

In [ ]:
import os
if os.path.exists("data/processed/jira_pairs.parquet"):
    jira = pipeline.layer2("jira")
    display(pd.DataFrame(jira["test"])[["scope", "method_label", "precision", "recall", "f1", "pr_auc"]])